# Antrenare reader pe Colab (T4)

Fine-tuning DistilBERT si BERT-base pe SQuAD v1.1 cu `src/qa/train.py`. Modelele, checkpoint-urile si rezultatele se urca pe Hugging Face Hub.

Inainte de rulare:
1. Runtime -> Change runtime type -> **T4 GPU**
2. In bara din stanga, iconita cu cheie (**Secrets**) -> Add new secret, cu "Notebook access" activat:
   - `HF_TOKEN` = token Hugging Face cu drept de scriere
   - `GH_TOKEN` = token GitHub read-only pentru repo (necesar doar cat timp repo-ul e privat)

   Daca un secret lipseste, celula respectiva il cere interactiv.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Cod si dependente

In [ ]:
import os
import subprocess


def get_secret(name, prompt):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    from getpass import getpass
    return getpass(prompt)


REPO = "Darius762/nlp-qa-system"
REPO_DIR = "/content/nlp-qa-system"

if os.path.exists(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    gh_token = get_secret("GH_TOKEN", "GitHub token (Enter daca repo-ul e public): ").strip()
    url = f"https://{gh_token}@github.com/{REPO}.git" if gh_token else f"https://github.com/{REPO}.git"
    result = subprocess.run(["git", "clone", url, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed: " + result.stderr.replace(gh_token or "<none>", "***"))

os.chdir(REPO_DIR)
!git log --oneline -1
!pip install -q -r requirements-colab.txt

## 2. Login Hugging Face

In [ ]:
from huggingface_hub import login, whoami

login(token=get_secret("HF_TOKEN", "HF write token: "))
print("logged in as", whoami()["name"])

## 3. Configuratie

`RESUME = True` doar daca sesiunea s-a deconectat in timpul antrenarii: reia de la ultimul checkpoint urcat pe Hub (`last-checkpoint`).

In [ ]:
HF_USER = "dariurs"
RESUME = False
resume_flag = "--resume-from-hub" if RESUME else ""

## 4. DistilBERT (~30-45 min)

In [ ]:
!python -m src.qa.train \
    --model-name distilbert-base-uncased \
    --epochs 2 --lr 3e-5 --batch-size 16 \
    --max-length 384 --stride 128 --seed 42 --fp16 \
    --push-to-hub --hub-model-id {HF_USER}/distilbert-base-uncased-squad {resume_flag}

## 5. BERT-base (~1-1.5 h)

In [ ]:
!python -m src.qa.train \
    --model-name bert-base-uncased \
    --epochs 2 --lr 3e-5 --batch-size 12 \
    --max-length 384 --stride 128 --seed 42 --fp16 \
    --push-to-hub --hub-model-id {HF_USER}/bert-base-uncased-squad {resume_flag}

## 6. Rezultate

Metricile sunt si pe Hub, in folderul `results/` al fiecarui model, de unde se descarca local.

In [ ]:
import json
from pathlib import Path

for path in sorted(Path("results").glob("reader_*_metrics.json")):
    m = json.loads(path.read_text())
    print(f"{path.name}: EM {m['exact_match']:.2f}  F1 {m['f1']:.2f}  "
          f"train {m['train']['train_runtime']:.0f}s on {m['device']}")